# 02 — Explore candidate features

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/a-quant-journey/blob/main/notebooks/teaching/02_construct_candidate_features.ipynb)

This notebook describes trailing indicators and cross-instrument structure in bounded daily observations. It uses training-only scaling, PCA and KMeans, then carries the fitted transformations to validation observations. These descriptive indicators and transformations are not automatically the three predictors used in Notebook 03.

**Google Colab:** Use the 2026.07 CPU runtime. On first execution, dependency installation may require a session restart. If prompted, restart the session and rerun the setup cell before continuing.

In [ ]:
# Colab only. Select a Python 3.12 CPU runtime.
import os

if (type(get_ipython()).__module__.startswith('google.colab.')
        and not os.environ.get('AQJ_COLAB_PREREQUISITE')):
    import runpy
    import subprocess
    from pathlib import Path

    branch = 'main'
    checkout = Path('/content/aqj-teaching') / branch
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--branch', branch, '--single-branch',
                        'https://github.com/FranQuant/a-quant-journey.git', str(checkout)], check=True)
    subprocess.run(['git', 'diff', '--quiet', 'HEAD'], cwd=checkout, check=True)
    runpy.run_path(str(checkout / 'scripts/colab_teaching.py'))['initialize'](2)

## 1. Data and setup

Set the repository paths and load the 12 instruments through **2021-12-28**. We use SPY’s observed closes for the trailing indicators and the multi-instrument panel for the later change matrix.

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src/quant_journey').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start within the teaching checkout or runner snapshot.')
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data/raw/tpq_eod_2014_2023.csv'
from src.quant_journey.execution.prediction_handoff import CUTOFF

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from src.quant_journey.data.tpq import INSTRUMENTS, load_tpq_columns_through, observed_sessions

cols = list(INSTRUMENTS)
prices = load_tpq_columns_through(DATA, cols, CUTOFF)
spy = prices['SPY'].dropna()

## 2. Trailing indicators: SMA, RSI and MACD

Calculate a trailing 20-observation simple moving average, a 14-session RSI using simple-average gains and losses (not Wilder smoothing), and the 12/26-span MACD from SPY closes. These summarize recent price movement and are descriptive examples; Notebook 03 uses a separate fixed set of three causal predictors.

In [ ]:
get_ipython().run_line_magic('matplotlib', 'inline')
ind = pd.DataFrame(index=spy.index)
ind['close'] = spy
ind['SMA_20'] = spy.rolling(20).mean()
delta = spy.diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
ind['RSI_14'] = 100 - 100 / (1 + gain / loss)
ind['MACD'] = spy.ewm(span=12, adjust=False).mean() - spy.ewm(span=26, adjust=False).mean()
ind[['close', 'SMA_20']].plot(
    figsize=(12, 4), title='SPY close and trailing 20-session mean'
)
plt.tight_layout()
plt.show()
ind['RSI_14'].plot(figsize=(12, 3), title='Trailing RSI(14)')
plt.axhline(70, color='r', ls='--')
plt.axhline(30, color='g', ls='--')
plt.tight_layout()
plt.show()
display(ind.tail())

**Interpretation.** The close and moving average show smoothed price direction; RSI summarizes recent up/down movement relative to reference levels 30 and 70; MACD is the difference between two exponential averages. They describe this SPY history and are not the Notebook 03 predictor set.

## 3. Prepare the change matrix

Calculate consecutive log changes for all 12 instruments. `.VIX` contributes changes in an implied-volatility index, not investment returns. Dates blank for both SPY and `.SPX` are omitted from U.S. instruments under the documented inferred calendar convention; simultaneous blanks do not prove an exchange closure. Other unexplained missing observations are not bridged. Drop rows with any missing change so every matrix row contains all 12 instruments. A row dated t is usable only after all 12 inputs are available. A shared date alone does not establish availability at the U.S. close. Split chronologically at 2020-01-02, then fit the standardizer on training rows only and apply it unchanged to validation rows.

In [ ]:
sessions = observed_sessions(prices)
logr = pd.DataFrame(index=prices.index, columns=cols, dtype=float)
for asset, asset_prices in sessions.items():
    logr[asset] = np.log(asset_prices / asset_prices.shift(1)).reindex(prices.index)
complete_log_returns = logr.dropna(how='any')
from src.quant_journey.execution.prediction_handoff import VALIDATION_START
cut = pd.Timestamp(VALIDATION_START)
Xtr = complete_log_returns.loc[complete_log_returns.index < cut]
Xva = complete_log_returns.loc[complete_log_returns.index >= cut]

## 4. Standardize observations and fit PCA

In [ ]:
scaler = StandardScaler().fit(Xtr)
Ztr = scaler.transform(Xtr)
Zva = scaler.transform(Xva)
pca = PCA(n_components=3, random_state=42).fit(Ztr)
train_pc = pca.transform(Ztr)
val_pc = pca.transform(Zva)
print('All-asset complete return rows:', len(complete_log_returns))
print('Training rows:', len(Xtr), 'validation rows:', len(Xva))
print('PCA explained-variance ratios:', pca.explained_variance_ratio_)
coefficients = pd.DataFrame(
    pca.components_.T, index=cols, columns=['PC1', 'PC2', 'PC3']
)
display(coefficients)
assert (len(Xtr), len(Xva)) == (1503, 502)
assert all(np.isfinite(x).all() for x in (Ztr, Zva, train_pc, val_pc))

**Interpretation.** After complete-case filtering, the matrix contains 2,005 rows: 1,503 training and 502 validation observations. The printed ratios describe variance in this standardized training matrix. The score plot below places days in PCA space; these are observations, not asset exposures.

## 5. Asset coefficients

The left panel plots daily scores on the first two components. The right panel places each instrument at its two component weights. Nearby instruments have similar coefficients in this sample. These are principal-axis weights. A common alternative loading convention multiplies each weight by the square root of its component eigenvalue.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(train_pc[:, 0], train_pc[:, 1], s=9, alpha=.35, label='train')
axes[0].scatter(val_pc[:, 0], val_pc[:, 1], s=9, alpha=.35, label='validation')
axes[0].set(xlabel='PC1', ylabel='PC2', title='Daily cross-instrument change observations')
axes[0].legend()
axes[1].scatter(coefficients['PC1'], coefficients['PC2'], s=35, zorder=3)
label_offsets = {
    'AAPL.O': (50, -34), 'MSFT.O': (-15, -52), 'INTC.O': (48, 20),
    'AMZN.O': (-48, 27), 'GS.N': (4, 40), '.SPX': (-58, -20),
    '.VIX': (-18, -30), 'SPY': (-58, 22), 'EUR=': (18, 18),
    'XAU=': (12, 27), 'GDX': (-38, -36), 'GLD': (22, -25),
}
for asset, row in coefficients.iterrows():
    axes[1].annotate(
        asset,
        xy=(row['PC1'], row['PC2']),
        xytext=label_offsets[asset],
        textcoords='offset points',
        fontsize=8,
        bbox={'boxstyle': 'round,pad=0.15', 'facecolor': 'white',
              'edgecolor': 'none', 'alpha': .82},
        arrowprops={'arrowstyle': '-', 'color': '.4', 'linewidth': .55,
                    'shrinkA': 2, 'shrinkB': 2},
        zorder=4,
    )
axes[1].axhline(0, color='grey', lw=.6)
axes[1].axvline(0, color='grey', lw=.6)
axes[1].margins(x=.20, y=.24)
axes[1].set(xlabel='PC1 coefficient', ylabel='PC2 coefficient',
            title='Instrument coefficients (train-fitted PCA)')
plt.tight_layout()
plt.show()

**Interpretation.** In the left panel, each point is a daily observation and color separates training from validation dates. In the right panel, each point is an instrument coefficient. Together the panels show where days lie in component space and how instruments align with the fitted components.

## 6. Cluster daily observations with KMeans

Fit three centers to training days in the full 12-dimensional standardized space, then assign each validation day to a fitted center. The plot shows training assignments projected onto PC1 and PC2. Cluster membership describes historical similarity and supplies no forecast.

In [ ]:
km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(Ztr)
train_labels = km.labels_
val_labels = km.predict(Zva)
display(pd.DataFrame({
    'train_cluster_counts': pd.Series(train_labels).value_counts().sort_index(),
    'validation_cluster_counts': pd.Series(val_labels).value_counts().sort_index(),
}))
plt.figure(figsize=(7, 5))
plt.scatter(train_pc[:, 0], train_pc[:, 1], c=train_labels, s=10, cmap='viridis')
plt.title('Daily training observations coloured by KMeans cluster')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()

**Interpretation.** Cluster counts show how the fixed partition assigns training days and validation days. The colors in the plot identify training-day assignments; they are descriptive groups, not targets, forecasts or a model-selection result.

## Conclusion

SMA, RSI and MACD describe SPY history. PCA summarizes cross-instrument variation and KMeans groups days. None of these indicators, PCA scores, component coefficients or cluster assignments enters Notebook 03’s three fixed predictors. See the teaching guide for numerical backend comparisons.